# Detecting "Death Due to Relapse" in Multilingual Clinical Notes

**A two-agent system built with Google's Agent Development Kit (ADK).**

You give the system a short clinical note written in any language (this project tests French, Spanish, English, German, Italian and Arabic). It answers one question:

Does the text say that the patient died of a relapse? Yes or no.

| Agent | Job |
|---|---|
| **TranslatorAgent** | Translates the note into English |
| **RelapseDeathClassifierAgent** | Reads the English text and answers `ANSWER: YES` or `ANSWER: NO` |



## Contents

| Section | What is built |
|---|---|
| 1 | Setup: install, API key, imports, constants |
| 2 | Test data: invented multilingual notes with expected answers |
| 3 | Agent 1: the Translator |
| 4 | Agent 2: the Classifier, with a custom tool |
| 5 | Connecting both agents into one pipeline (`SequentialAgent`) |
| 6 | Sessions, persistent storage, state and context compaction |
| 7 | Long-term memory across sessions |
| 8 | Human-in-the-loop approval before an action |
| 9 | Observability: logs, plugins and metrics |
| 10 | Packaging the agent as an ADK project (`adk web`) |
| 11 | Evaluation: labeled accuracy test and `adk eval` |
| 12 | Deployment to Vertex AI Agent Engine |

**Important notes**

- **This is a screening helper, not a clinical decision tool.** A person must verify any result that matters.
- **Privacy:** use only de-identified text. Do not send real patient data to a model or cloud service unless your organisation has approved it. Every note in this notebook is invented.
- **Saved outputs:** the outputs shown below come from a real run, so the results can be read without running anything.
- **Rate limits:** if you do rerun it, run cells one at a time and in order, not "Run all", to avoid `429` (too many requests) errors from the model API.

---


## Section 1: Setup

### 1.1: Install dependencies

Install the ADK package once (remove the `#` to run it). Skip this cell if it is already installed.


In [1]:
# !pip install -q google-adk

### 1.2: Configure the Gemini API key

The notebook uses the [Gemini API](https://ai.google.dev/gemini-api/), which needs an API key from [Google AI Studio](https://aistudio.google.com/app/api-keys).

Create a file named `.env` next to the notebook with one line, `GOOGLE_API_KEY=your_key_here`. The cell below reads that file, so the key is never written inside the notebook. Never share or publish the `.env` file.


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 1.3: Import ADK components

Import everything needed. The groups are:

- **Agents:** `LlmAgent` (one agent powered by a language model) and `SequentialAgent` (runs agents in a fixed order).
- **Runtime:** `Runner` / `InMemoryRunner` (execute agents) and `App` (bundles an agent with extra settings such as compaction, plugins and resumability).
- **State:** session services (short-term) and a memory service (long-term).
- **Tools and plugins:** `ToolContext`, `FunctionTool`, `preload_memory`, and the plugin classes used for observability.


In [ ]:
import os
import asyncio
import json
import logging
import re
import subprocess
import sys
import time
import uuid
import warnings
from pathlib import Path

from google.genai import types

from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.apps.app import App, EventsCompactionConfig, ResumabilityConfig
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner, Runner
from google.adk.sessions import DatabaseSessionService, InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import ToolContext, preload_memory
from google.adk.tools.function_tool import FunctionTool

# Observability 
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.plugins.base_plugin import BasePlugin
from google.adk.plugins.logging_plugin import LoggingPlugin
from google.adk.tools.base_tool import BaseTool


warnings.filterwarnings("ignore")
print("ADK components imported successfully.")

ADK components imported successfully.


### 1.4: Configure retry options and constants

Model APIs sometimes return temporary errors (rate limits, overloaded servers). `HttpRetryOptions` makes ADK retry automatically, waiting longer after each failure.

The constants below are used throughout: one model name, one app name and one user id.


In [59]:
retry_config = types.HttpRetryOptions(
    attempts=5, 
    exp_base=7, 
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  
)

MODEL_NAME = "gemini-3.5-flash-lite"
APP_NAME = "relapse_screening_app"
USER_ID = "clinician_demo"

print("Configuration ready.")

Configuration ready.


## Section 2: Test data

Before building anything, I write down test notes and the answer expected for each one, so the agents can be checked later.

All notes are **invented** and de-identified. They are written in six languages and designed to trip up a careless system:

| Case | Language | What makes it tricky |
|---|---|---|
| `relapse_death_explicit` | French | The easy positive case |
| `death_other_cause_no_relapse` | Spanish | Death, but the relapse is negated (no sign of relapse) |
| `relapse_but_alive` | English | Relapse without death |
| `death_cause_not_stated` | German | Death with no stated cause |
| `recurrence_synonym` | Italian | Says *recidiva* (recurrence) instead of relapse |
| `negated_relapse` | Arabic | "Did **not** die of a relapse" |
| `died_after_relapse_no_causal_link` | French | Relapse and death both appear, but no causal link is stated |

**The rule used:** answer `YES` only if the text **explicitly says the patient died** and **attributes the death to a relapse or recurrence**. Everything else is `NO`, including "cause not stated". The last case is debatable, and a clinician might disagree. That is why the rule is written down and placed in the agent's instruction.


In [58]:
SAMPLES = [
    {
        "id": "relapse_death_explicit",
        "expected": "YES",
        "Original": "Le patient, âgé de 58 ans, est décédé le 12 mars des suites d'une rechute de sa leucémie myéloïde aiguë.",
    },
    {
        "id": "death_other_cause_no_relapse",
        "expected": "NO",
        "Original": "La paciente falleció a causa de una embolia pulmonar. No se observó ningún signo de recaída durante el último control.",
    },
    {
        "id": "relapse_but_alive",
        "expected": "NO",
        "Original": "The patient experienced a relapse in January. He is currently undergoing chemotherapy, and his condition is stable.",
    },
    {
        "id": "death_cause_not_stated",
        "expected": "NO",
        "Original": "Der Tod trat am 3. April zu Hause ein. Die Todesursache ist in den Unterlagen nicht angegeben.",
    },
    {
        "id": "recurrence_synonym",
        "expected": "YES",
        "Original": "Paziente deceduto in seguito a una recidiva tumorale epatica, nonostante una seconda linea di trattamento.",
    },
    {
        "id": "negated_relapse",
        "expected": "NO",
        "Original": "لم يتوفَّ المريض بسبب انتكاسة، بل جراء إصابته بالتهاب رئوي حاد أثناء وجوده في المستشفى.",
    },
    {
        "id": "died_after_relapse_no_causal_link",
        "expected": "NO",
        "Original": "Le patient a rechuté en septembre et est décédé quelques semaines plus tard. Le certificat de décès ne mentionne pas de cause.",
    },
]

print(f"{len(SAMPLES)} test notes defined.")
for s in SAMPLES:
    print(f"  • {s['id']:<38} expected: {s['expected']}")

7 test notes defined.
  • relapse_death_explicit                 expected: YES
  • death_other_cause_no_relapse           expected: NO
  • relapse_but_alive                      expected: NO
  • death_cause_not_stated                 expected: NO
  • recurrence_synonym                     expected: YES
  • negated_relapse                        expected: NO
  • died_after_relapse_no_causal_link      expected: NO


---
## Section 3: Agent 1, the Translator

The first agent has one job: translate the clinical text into English.

**Why the instruction matters:** the next agent decides based on this translation, so a careless translation would silently change the final answer. The instruction therefore says: be faithful, keep negations, use standard medical terms, and output only the translation.

**`output_key="english_text"`** tells ADK to save this agent's final answer in the **session state** under that name. The next agent reads it from there.


In [ ]:
TRANSLATOR_INSTRUCTION = """You are a professional medical translator.
Translate the clinical text from different languages written by the user into English. If the text is already in English, return it as is.

Rules:
- Translate faithfully and completely. Do NOT summarize, interpret, add or omit information.
- Keep negations exactly as written (for example 'pas de rechute' -> 'no relapse').
- Use standard English medical terms ('relapse', 'recurrence', 'death',  'died').
- Output ONLY the English translation, with no introduction, notes or quotation marks."""


def build_translator() -> LlmAgent:
    return LlmAgent(
        name="TranslatorAgent",
        model=Gemini(model=MODEL_NAME, retry_options=retry_config),
        description="Translates clinical text from different languages into faithful English.",
        instruction=TRANSLATOR_INSTRUCTION,
        output_key="english_text",  
    )


print("Translator defined.")

Translator defined.


### 3.1: Run the translator on its own

`InMemoryRunner` runs an agent quickly while prototyping, and `run_debug()` creates a session and prints the conversation.

Check that the negation ("No signs of relapse") and the medical terms are preserved.


In [11]:
translator_runner = InMemoryRunner(agent=build_translator())

_ = await translator_runner.run_debug(SAMPLES[1]["Original"], session_id="test_session_1")  


 ### Created new session: test_session_1

User > La paciente falleció a causa de una embolia pulmonar. No se observó ningún signo de recaída durante el último control.
TranslatorAgent > The patient died of a pulmonary embolism. No signs of relapse were observed during the last follow-up.


---
## Section 4: Agent 2, the Classifier, with a custom tool

### 4.1: Why give the classifier a tool?

The tool `find_relapse_death_evidence` scans the English text with simple patterns and returns the sentences that mention **death**, **relapse or recurrence**, **both together**, and relapse sentences that **may be negated** ("no sign of relapse"). The language model then uses this structured evidence to make the final decision.

### 4.2: Tool design practices applied

| Practice | How the tool does it |
|---|---|
| **Clear docstring** (the model reads it to decide when and how to call the tool) | States what the tool does and what it returns |
| **Type hints** | `-> dict` return type |
| **Dictionary return with a `status`** | `{"status": "success", ...}` or `{"status": "error", "error_message": ...}` |
| **Error handling** | Returns an error dictionary if there is no text in state |
| **`ToolContext`** | Gives the tool access to the session state |

**Design choice:** the tool takes **no text argument**. It reads `english_text` from the session state, where the translator already put it. This avoids the model re-typing (and possibly altering) the text, and it makes the tool call identical every time, which matters later when the tool calls are compared with an expected "trajectory" in the evaluation.


In [ ]:
DEATH_PATTERN = re.compile(
    r"\b(died|dead|death|deceased|passed away|succumbed|fatal(?:ly)?)\b", re.IGNORECASE
)
RELAPSE_PATTERN = re.compile(r"\b(relaps\w*|recurr\w*)\b", re.IGNORECASE)
NEGATION_PATTERN = re.compile(
    r"\b(no|not|without|never|free of|absence of|absent)\b", re.IGNORECASE
)


def find_relapse_death_evidence(tool_context: ToolContext) -> dict:
    """Finds the sentences of the translated clinical text that mention death and relapse.

    Reads the English translation that the TranslatorAgent saved in session state
    under the key 'english_text'. Takes no arguments.

    Returns:
        Dictionary with status and evidence.
        Success: {"status": "success", "death_sentences": [...], "relapse_sentences": [...],
                  "sentences_with_both": [...], "relapse_sentences_possibly_negated": [...]}
        Error: {"status": "error", "error_message": "..."}
    """
    text = tool_context.state.get("english_text")
    if not text:
        return {
            "status": "error",
            "error_message": "No English text found in session state under 'english_text'.",
        }

    sentences = re.split(r"(?<=[.!?;])\s+", text.strip())
    death = [s for s in sentences if DEATH_PATTERN.search(s)]
    relapse = [s for s in sentences if RELAPSE_PATTERN.search(s)]
    both = [s for s in death if RELAPSE_PATTERN.search(s)]
    negated = [s for s in relapse if NEGATION_PATTERN.search(s)]

    return {
        "status": "success",
        "death_sentences": death,
        "relapse_sentences": relapse,
        "sentences_with_both": both,
        "relapse_sentences_possibly_negated": negated,
    }


print("Tool defined.")

Tool defined.


### 4.3: Test the tool without any language model

A tool is plain Python, so it can be tested directly and gives the same result every time. A tiny stand-in for `ToolContext`, with only a `.state` dictionary, is enough. The third call shows the error case (no text in state).


In [13]:
class _StubToolContext:
    """Minimal stand-in for ToolContext, just for testing the tool locally."""

    def __init__(self, state: dict):
        self.state = state


print(
    find_relapse_death_evidence(
        _StubToolContext({"english_text": "The patient died of a relapse of acute leukemia."})
    )
)
print()
print(
    find_relapse_death_evidence(
        _StubToolContext(
            {"english_text": "The patient died of a pulmonary embolism. No sign of relapse was observed."}
        )
    )
)
print()
print(find_relapse_death_evidence(_StubToolContext({})))

{'status': 'success', 'death_sentences': ['The patient died of a relapse of acute leukemia.'], 'relapse_sentences': ['The patient died of a relapse of acute leukemia.'], 'sentences_with_both': ['The patient died of a relapse of acute leukemia.'], 'relapse_sentences_possibly_negated': []}

{'status': 'success', 'death_sentences': ['The patient died of a pulmonary embolism.'], 'relapse_sentences': ['No sign of relapse was observed.'], 'sentences_with_both': [], 'relapse_sentences_possibly_negated': ['No sign of relapse was observed.']}

{'status': 'error', 'error_message': "No English text found in session state under 'english_text'."}


### 4.4: Define the classifier agent

- `{english_text}` is a **state placeholder**: ADK replaces it with the translator's output before the model sees the instruction.
- The agent must call the tool first, then decide using the text itself.
- The output is **exactly one line**, `ANSWER: YES` or `ANSWER: NO`. This makes the answer easy to read by code and to compare in the evaluation (Section 11).
- `output_key="relapse_verdict"` stores the answer in the session state too.


In [ ]:
CLASSIFIER_INSTRUCTION = """You are a careful clinical text screener.

The English clinical text to analyse is:
{english_text}

Question: does this text state that the patient died of a relapse (or recurrence) of their disease?

Steps:
1. Call the `find_relapse_death_evidence` tool (it takes no arguments) to get the sentences that mention death and relapse.
2. Decide using the text above, with the tool output as help.

Answer YES only if the text explicitly says the patient died AND attributes the death to a relapse or recurrence.
Answer NO in every other case, including:
- the patient is alive,
- the patient died of another cause,
- the cause of death is not stated,
- a relapse is mentioned but not stated as the cause of death,
- the relapse is negated (for example 'no sign of relapse', 'did not die of a relapse').

Respond with exactly one line and nothing else:
ANSWER: YES
or
ANSWER: NO"""


def build_classifier() -> LlmAgent:
    return LlmAgent(
        name="RelapseDeathClassifierAgent",
        model=Gemini(model=MODEL_NAME, retry_options=retry_config),
        description="Answers whether an English clinical text says the patient died of a relapse.",
        instruction=CLASSIFIER_INSTRUCTION,
        tools=[find_relapse_death_evidence],
        output_key="relapse_verdict",
    )


print("Classifier factory defined.")

Classifier factory defined.


---
## Section 5: Connect the two agents

A **`SequentialAgent`** is an assembly line: it runs its sub-agents in exactly the order listed. No language model "manager" decides the order, so the flow is predictable.

```
TranslatorAgent --(english_text)--> RelapseDeathClassifierAgent --(relapse_verdict)--> answer
```

The hand-off works through the **session state**: the translator writes `english_text` (through `output_key`), and the classifier reads it (through the `{english_text}` placeholder and through the tool).


In [ ]:
def build_pipeline(name: str = "RelapseScreeningPipeline", **kwargs) -> SequentialAgent:
    """Creates a fresh translator -> classifier pipeline (extra kwargs go to SequentialAgent)."""
    return SequentialAgent(
        name=name,
        description="Translates a multilingual clinical note and says whether the patient died of a relapse.",
        sub_agents=[build_translator(), build_classifier()],
        **kwargs,
    )


pipeline_runner = InMemoryRunner(agent=build_pipeline())
print("Pipeline created.")

Pipeline created.


### 5.1: Run the full pipeline

Watch the output: first the translator's English text, then the tool call (shown as a warning about a non-text part, which is the function call), then the one-line answer. Here the expected answer is **YES**.


In [16]:
_ = await pipeline_runner.run_debug(SAMPLES[0]["Original"])


 ### Created new session: debug_session_id

User > Le patient, âgé de 58 ans, est décédé le 12 mars des suites d'une rechute de sa leucémie myéloïde aiguë.
TranslatorAgent > The 58-year-old patient died on March 12 as a result of a relapse of his acute myeloid leukemia.


RelapseDeathClassifierAgent > ANSWER: YES


### 5.2: Try a tricky case

A relapse is mentioned but the patient is alive. The expected answer is **NO**.


In [17]:
_ = await pipeline_runner.run_debug(SAMPLES[2]["Original"])  


 ### Continue session: debug_session_id

User > The patient experienced a relapse in January. He is currently undergoing chemotherapy, and his condition is stable.
TranslatorAgent > The patient experienced a relapse in January. He is currently undergoing chemotherapy, and his condition is stable.
RelapseDeathClassifierAgent > ANSWER: NO


---
## Section 6: Sessions, persistence and state

Until now `run_debug()` hid the session handling. A **session** is one conversation: it stores every message, model output and tool call, plus a small dictionary called **state**.

### 6.1: The `App` with context compaction

The pipeline is wrapped in an **`App`** with **context compaction** switched on: after a few turns ADK summarises old events so long sessions stay cheap and fast. With one note per session it will rarely trigger, but it is the right default when someone keeps chatting in the same session.

The session service below stores sessions in a **SQLite database file**, so they survive after the code stops. The `build_app` function is reused in later sections, so it accepts an optional callback and optional plugins.


In [ ]:
def build_app(after_agent_callback=None, plugins=None) -> App:
    """Builds the app: pipeline + context compaction (+ optional callback and plugins)."""
    pipeline_kwargs = {}
    if after_agent_callback:
        pipeline_kwargs["after_agent_callback"] = after_agent_callback
    return App(
        name=APP_NAME,
        root_agent=build_pipeline(**pipeline_kwargs),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=3,  
            overlap_size=1, 
        ),
        plugins=plugins or [],
    )


if os.path.exists("relapse_sessions.db"):
    os.remove("relapse_sessions.db")

session_service = DatabaseSessionService(db_url="sqlite+aiosqlite:///relapse_sessions.db")
runner = Runner(app=build_app(), session_service=session_service)

print("App + persistent session service ready.")
print(f"App: {APP_NAME}   - Session service: {session_service.__class__.__name__}")

App + persistent session service ready.
App: relapse_screening_app   - Session service: DatabaseSessionService


### 6.2: A helper to screen one note

This helper creates a session, sends the original note, reads the events as they stream in, and returns **the final answer** and **the final session state**.

Two details:

- The pipeline has two agents, so there are two "final responses": the translator's and the classifier's. The helper keeps the **last** one (the classifier's).
- Compaction events are skipped, so a summary can never be mistaken for the answer.


In [ ]:
async def screen_note(runner_instance, svc, text: str, session_id: str):
    """Runs one note in its own session. Returns (final_answer_text, session_state)."""
    app_name = runner_instance.app_name
    try:
        session = await svc.create_session(app_name=app_name, user_id=USER_ID, session_id=session_id)
    except Exception:
        session = await svc.get_session(app_name=app_name, user_id=USER_ID, session_id=session_id)

    message = types.Content(role="user", parts=[types.Part(text=text)])
    final_text = None
    async for event in runner_instance.run_async(
        user_id=USER_ID, session_id=session.id, new_message=message
    ):
        if getattr(event.actions, "compaction", None):
            continue  # ignore context-compaction summaries
        if event.is_final_response() and event.content and event.content.parts:
            part_text = event.content.parts[0].text
            if part_text and part_text != "None":
                final_text = part_text.strip()

    session = await svc.get_session(app_name=app_name, user_id=USER_ID, session_id=session.id)
    return final_text, dict(session.state)


print("Helper defined.")

Helper defined.


### 6.3: Screen every note, one session each

Each note gets its own session id (`note-<id>`). For each note the cell prints the original text, the English translation (read from **state**) and the answer next to the expected one.


In [ ]:
results = []
for sample in SAMPLES:
    answer, state = await screen_note(runner, session_service, sample["Original"], f"note-{sample['id']}")
    results.append((sample, answer, state))
    print(f"\n {sample['id']}  (expected {sample['expected']})")
    print(f"   Or : {sample['Original']}")
    print(f"   EN : {state.get('english_text')}")
    print(f"   →  : {answer}")


relapse_death_explicit  (expected YES)
   Or : Le patient, âgé de 58 ans, est décédé le 12 mars des suites d'une rechute de sa leucémie myéloïde aiguë.
   EN : The 58-year-old patient died on March 12 as a result of a relapse of his acute myeloid leukemia.
   →  : ANSWER: YES

death_other_cause_no_relapse  (expected NO)
   Or : La paciente falleció a causa de una embolia pulmonar. No se observó ningún signo de recaída durante el último control.
   EN : The patient died due to a pulmonary embolism. No sign of relapse was observed during the last follow-up.
   →  : ANSWER: NO

relapse_but_alive  (expected NO)
   Or : The patient experienced a relapse in January. He is currently undergoing chemotherapy, and his condition is stable.
   EN : The patient experienced a relapse in January. He is currently undergoing chemotherapy, and his condition is stable.
   →  : ANSWER: NO

death_cause_not_stated  (expected NO)
   Or : Der Tod trat am 3. April zu Hause ein. Die Todesursache ist in den Unt

### 6.4: Look inside the database

The sessions were saved. Every event (user message, model output, tool call and tool result) is a row in the `events` table, which also works as a raw audit trail.


In [21]:
import sqlite3


def peek_database(limit: int = 8):
    with sqlite3.connect("relapse_sessions.db") as connection:
        cursor = connection.cursor()
        rows = cursor.execute("select session_id, author from events").fetchall()
        print(f"Total events stored: {len(rows)}   (showing first {limit})")
        for row in rows[:limit]:
            print("  ", row)


peek_database()

Total events stored: 35   (showing first 8)
   ('note-relapse_death_explicit', 'user')
   ('note-relapse_death_explicit', 'TranslatorAgent')
   ('note-relapse_death_explicit', 'RelapseDeathClassifierAgent')
   ('note-relapse_death_explicit', 'RelapseDeathClassifierAgent')
   ('note-relapse_death_explicit', 'RelapseDeathClassifierAgent')
   ('note-death_other_cause_no_relapse', 'user')
   ('note-death_other_cause_no_relapse', 'TranslatorAgent')
   ('note-death_other_cause_no_relapse', 'RelapseDeathClassifierAgent')


### 6.5: Session isolation

A new session is started and asked about the previous patient's note. A well-isolated system must know nothing about it. The classifier simply treats the question as text to analyse and answers `NO`, and the state does not contain the old note.


In [22]:
answer, state = await screen_note(
    runner, session_service, "Que disait la note du patient précédent ?", "isolated-session"
)
print("Answer:", answer)
print("State :", state)

Answer: ANSWER: NO
State : {'english_text': "What did the previous patient's note say?", 'relapse_verdict': 'ANSWER: NO'}


---
## Section 7: Long-term memory

Sessions are **short-term** (one conversation). **Memory** is **long-term** and searchable **across sessions**. For this project, memory lets a reviewer ask later, "Which of the notes we screened were answered YES?", even though each note lived in a different session.

The workflow has three steps:

1. **Initialize:** create a memory service and give it to the `Runner`.
2. **Ingest:** move session data into memory with `add_session_to_memory()` (automated here with a callback).
3. **Retrieve:** give an agent a memory tool (`preload_memory`) so it can search memory.

### 7.1: Initialize memory and automate ingestion

The `after_agent_callback` runs automatically after the pipeline finishes a note and saves the session to memory, so no manual call is needed.


In [ ]:
memory_service = InMemoryMemoryService()


async def auto_save_to_memory(callback_context):
    """Automatically save the finished session to long-term memory."""
    await callback_context._invocation_context.memory_service.add_session_to_memory(
        callback_context._invocation_context.session
    )


memory_runner = Runner(
    app=build_app(after_agent_callback=auto_save_to_memory),
    session_service=session_service,
    memory_service=memory_service, 
)
print("Pipeline runner with automatic memory saving created.")

Pipeline runner with automatic memory saving created.


### 7.2: Screen two notes (they are saved to memory automatically)


In [24]:
for sample in [SAMPLES[0], SAMPLES[2]]: 
    answer, _ = await screen_note(memory_runner, session_service, sample["Original"], f"mem-{sample['id']}")
    print(f"{sample['id']}: {answer}")

relapse_death_explicit: ANSWER: YES
relapse_but_alive: ANSWER: NO


### 7.3: A "case history" agent that can search memory

A separate agent with the **`preload_memory`** tool, which automatically searches memory before each turn. It must use the **same app name and user id** as the pipeline, because memory is scoped by them.

It is told to answer **only** from what memory returns, so it does not invent cases. The question is asked in a brand-new session, so memory is the only way to answer it.


In [25]:
case_history_agent = LlmAgent(
    name="CaseHistoryAgent",
    model=Gemini(model=MODEL_NAME, retry_options=retry_config),
    instruction="""You help clinicians review notes that were already screened.
Memory contains the original notes, their English translations and the screening ANSWER (YES/NO).
Answer ONLY from the recalled memory. If memory has nothing relevant, say you have no record. Never guess.""",
    tools=[preload_memory],
)

history_runner = Runner(
    agent=case_history_agent,
    app_name=APP_NAME,  
    session_service=session_service,
    memory_service=memory_service,
)

# A NEW session: the only way to answer is through long-term memory.
answer, _ = await screen_note(
    history_runner,
    session_service,
    "Among the notes already screened, which patient died of a relapse? Summarize it in English.",
    "history-session-1",
)
print(answer)

Based on the records, the 58-year-old patient died on March 12 as a result of a relapse of acute myeloid leukemia.


---
## Section 8: Human-in-the-loop approval

Sometimes an answer triggers an action with consequences, for example adding a case to a "death due to relapse" mortality-review registry. For that, a **clinician must approve** before anything is written.

A tool can **pause** the agent, wait for a human decision, then **resume**:

```
Agent calls tool --> tool asks for confirmation --> PAUSE --> human approves or rejects --> RESUME --> tool completes
```

The pieces:

1. A tool that calls `ToolContext.request_confirmation()` on the first call and checks `tool_context.tool_confirmation` when resumed.
2. An agent that uses this tool.
3. A **resumable `App`** (`ResumabilityConfig(is_resumable=True)`) so ADK can save and restore the paused state.
4. Workflow code that detects the special `adk_request_confirmation` event and resumes with the **same `invocation_id`**.


In [ ]:
def register_relapse_death_case(case_id: str, tool_context: ToolContext) -> dict:
    """Registers a case in the mortality-review registry. ALWAYS requires clinician approval.

    Args:
        case_id: Identifier of the case or note (never a patient name).

    Returns:
        Dictionary with the registration status ('pending', 'registered' or 'rejected').
    """
    if not tool_context.tool_confirmation:
        tool_context.request_confirmation(
            hint=f"Register case '{case_id}' as 'death due to relapse' in the review registry. Approve?",
            payload={"case_id": case_id},
        )
        return {"status": "pending", "message": f"Registration of {case_id} requires clinician approval."}

    if tool_context.tool_confirmation.confirmed:
        return {"status": "registered", "case_id": case_id, "message": f"Case {case_id} registered."}
    return {"status": "rejected", "case_id": case_id, "message": f"Registration of {case_id} was rejected."}


registry_agent = LlmAgent(
    name="registry_agent",
    model=Gemini(model=MODEL_NAME, retry_options=retry_config),
    instruction="""You are a registry assistant.
When asked to register a case, call the register_relapse_death_case tool with the case id.
If the status is 'pending', tell the user approval is required.
After the final result, state clearly whether the case was registered or rejected.""",
    tools=[FunctionTool(func=register_relapse_death_case)],
)

registry_app = App(
    name="relapse_registry_app",
    root_agent=registry_agent,
    resumability_config=ResumabilityConfig(is_resumable=True), 
)

registry_session_service = InMemorySessionService()
registry_runner = Runner(app=registry_app, session_service=registry_session_service)
print("Registry agent, resumable app and runner created.")

Registry agent, resumable app and runner created.


### 8.1: The workflow helpers

Three small helpers: one detects that the agent paused, one prints the agent's text, and one builds the human's answer as a `FunctionResponse`.


In [ ]:
def check_for_approval(events):
    """Returns approval details if the agent paused for confirmation, else None."""
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.function_call and part.function_call.name == "adk_request_confirmation":
                    return {"approval_id": part.function_call.id, "invocation_id": event.invocation_id}
    return None


def print_agent_text(events):
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent > {part.text}")


def create_approval_response(approval_info, approved: bool):
    response = types.FunctionResponse(
        id=approval_info["approval_id"],
        name="adk_request_confirmation",
        response={"confirmed": approved},
    )
    return types.Content(role="user", parts=[types.Part(function_response=response)])


print("Helpers defined.")

Helpers defined.


### 8.2: Run the approval workflow

1. Send the request. The tool pauses.
2. Detect the pause.
3. A (simulated) clinician decides.
4. Resume with the same `invocation_id`.

In a real application step 3 would be a button in a user interface. Here `approve=True/False` simulates it. The first case is approved and the second is rejected.


In [ ]:
async def run_registry_workflow(query: str, approve: bool):
    print(f"\n{'=' * 60}\nUser > {query}\n")
    session_id = f"reg_{uuid.uuid4().hex[:8]}"
    await registry_session_service.create_session(
        app_name="relapse_registry_app", user_id=USER_ID, session_id=session_id
    )
    message = types.Content(role="user", parts=[types.Part(text=query)])

    events = []
    async for event in registry_runner.run_async(user_id=USER_ID, session_id=session_id, new_message=message):
        events.append(event)

    approval_info = check_for_approval(events)

    if approval_info:
        print(f"Paused for approval. Clinician decision: {'APPROVE' if approve else 'REJECT'}\n")
        # STEP 3: resume — the SAME invocation_id tells ADK to resume instead of restart.
        resumed = []
        async for event in registry_runner.run_async(
            user_id=USER_ID,
            session_id=session_id,
            new_message=create_approval_response(approval_info, approve),
            invocation_id=approval_info["invocation_id"],
        ):
            resumed.append(event)
        print_agent_text(resumed)
    else:
        print_agent_text(events)


await run_registry_workflow("Register case NOTE-0001 as a death due to relapse.", approve=True)
await run_registry_workflow("Register case NOTE-0002 as a death due to relapse.", approve=False)


User > Register case NOTE-0001 as a death due to relapse.

Paused for approval. Clinician decision: APPROVE

Agent > The case NOTE-0001 has been registered successfully in the mortality-review registry.

User > Register case NOTE-0002 as a death due to relapse.

Paused for approval. Clinician decision: REJECT

Agent > The case NOTE-0002 was rejected for registration in the mortality-review registry.


---
## Section 9: Observability: logs, traces and metrics

When the agent answers `NO` instead of the expected `YES`, **why**? Observability shows the decision step by step: what the translator produced, what the tool returned, and what the model said.

The three parts are **logs** (what happened), **traces** (why) and **metrics** (how well, in total). Which one to use:

| Situation | Use |
|---|---|
| Development debugging | `adk web --log_level DEBUG` |
| Standard production logging | ADK's built-in **`LoggingPlugin`** |
| Custom needs | Your own **plugin** made of **callbacks** |

### 9.1: Configure DEBUG logging to a file


In [ ]:
for log_file in ["logger.log"]:
    if os.path.exists(log_file):
        os.remove(log_file)

logging.basicConfig(
    filename="logger.log",
    level=logging.DEBUG,
    format="%(filename)s:%(lineno)s %(levelname)s:%(message)s",
    force=True,
)
print("Logging configured (DEBUG level → logger.log)")

Logging configured (DEBUG level → logger.log)


### 9.2: A custom plugin that counts what the pipeline does

A **plugin** is custom code that runs automatically at points in the agent's life cycle. It is made of **callbacks** (before or after an agent, a tool, a model call, and so on).

`PipelineMetricsPlugin` counts agent runs, model requests and tool calls, and records which tools were called. These are simple **metrics**.


In [ ]:
class PipelineMetricsPlugin(BasePlugin):
    """Counts agent runs, LLM requests and tool calls across the whole pipeline."""

    def __init__(self) -> None:
        super().__init__(name="pipeline_metrics")
        self.agent_count = 0
        self.llm_request_count = 0
        self.tool_calls: list[str] = []

    async def before_agent_callback(self, *, agent, callback_context: CallbackContext) -> None:
        self.agent_count += 1

    async def before_model_callback(self, *, callback_context: CallbackContext, llm_request: LlmRequest) -> None:
        self.llm_request_count += 1

    async def before_tool_callback(self, *, tool: BaseTool, tool_args: dict, tool_context: ToolContext):
        self.tool_calls.append(tool.name)
        logging.info(f"[PipelineMetrics] tool called: {tool.name} args={tool_args}")
        return None  # None = let the tool run normally

    def summary(self) -> dict:
        return {
            "agent_runs": self.agent_count,
            "llm_requests": self.llm_request_count,
            "tool_calls": len(self.tool_calls),
            "tools_used": sorted(set(self.tool_calls)),
        }


print("Plugin defined.")

Plugin defined.


### 9.3: Add `LoggingPlugin` and the custom plugin, then run a note

`LoggingPlugin` prints a full trace: the user message, each agent, each model request and response, and each tool call and its result, with timings. Plugins are attached through the **`App`**.

In the output, look for the translator's request and response, the **tool call** `find_relapse_death_evidence` with its returned evidence, and then the final answer.


In [32]:
metrics = PipelineMetricsPlugin()
observed_runner = Runner(
    app=build_app(plugins=[LoggingPlugin(), metrics]),
    session_service=session_service,
    memory_service=memory_service,
)

answer, _ = await screen_note(observed_runner, session_service, SAMPLES[4]["Original"], "observed-1")
print("\nFinal answer:", answer)

[logging_plugin] USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-df45f589-53ee-43ec-ab30-47402b32c790
[logging_plugin]    Session ID: observed-1
[logging_plugin]    User ID: clinician_demo
[logging_plugin]    App Name: relapse_screening_app
[logging_plugin]    Root Agent: RelapseScreeningPipeline
[logging_plugin]    User Content: text: 'Paziente deceduto in seguito a una recidiva tumorale epatica, nonostante una seconda linea di trattamento.'
[logging_plugin] INVOCATION STARTING
[logging_plugin]    Invocation ID: e-df45f589-53ee-43ec-ab30-47402b32c790
[logging_plugin]    Starting Agent: RelapseScreeningPipeline
[logging_plugin] AGENT STARTING
[logging_plugin]    Agent Name: RelapseScreeningPipeline
[logging_plugin]    Invocation ID: e-df45f589-53ee-43ec-ab30-47402b32c790
[logging_plugin] AGENT STARTING
[logging_plugin]    Agent Name: TranslatorAgent
[logging_plugin]    Invocation ID: e-df45f589-53ee-43ec-ab30-47402b32c790
[logging_plugin] LLM REQUEST
[logging_plugin]    Mode

### 9.4: Read the metrics and the log file

The metrics give a quick summary. The log file keeps the DEBUG details (such as the full prompts sent to the model) for debugging later: symptom, then logs, then root cause, then fix. The last lines shown here are low-level database messages, which shows how detailed the DEBUG level is.


In [33]:
print("Metrics:", metrics.summary())
print("\nLast lines of logger.log:\n")
with open("logger.log", encoding="utf-8") as f:
    for line in f.readlines()[-12:]:
        print(line.rstrip()[:200])

Metrics: {'agent_runs': 3, 'llm_requests': 3, 'tool_calls': 1, 'tools_used': ['find_relapse_death_evidence']}

Last lines of logger.log:

core.py:104 DEBUG:executing functools.partial(<built-in method cursor of sqlite3.Connection object at 0x00000282297B8C70>)
core.py:106 DEBUG:operation functools.partial(<built-in method cursor of sqlite3.Connection object at 0x00000282297B8C70>) completed
core.py:104 DEBUG:executing functools.partial(<built-in method execute of sqlite3.Cursor object at 0x000002822987F1C0>, 'SELECT user_states.app_name AS user_states_app_name, user_states.user_id AS use
core.py:106 DEBUG:operation functools.partial(<built-in method execute of sqlite3.Cursor object at 0x000002822987F1C0>, 'SELECT user_states.app_name AS user_states_app_name, user_states.user_id AS use
core.py:104 DEBUG:executing functools.partial(<built-in method fetchall of sqlite3.Cursor object at 0x000002822987F1C0>)
core.py:106 DEBUG:operation functools.partial(<built-in method fetchall of sqlite3.

---
## Section 10: Package the agent as an ADK project

The ADK command-line tools (`adk web`, `adk eval`, `adk deploy`) need the agent as a **folder with an `agent.py`** that exposes a variable called **`root_agent`**.

The folder is written once. It contains the same translator, tool and classifier, assembled into the pipeline.

### 10.1: Create the project folder

`__init__.py` makes the folder importable. The `.env` file is copied so the command-line tools have the API key (**do not share or commit this folder**).


In [34]:
import shutil
from pathlib import Path

Path("relapse_agent").mkdir(exist_ok=True)
Path("relapse_agent/__init__.py").write_text("from . import agent\n")

shutil.copy(".env", "relapse_agent/.env")
print("Created relapse_agent/__init__.py and copied .env")

Created relapse_agent/__init__.py and copied .env


### 10.2: Write `agent.py`

`%%writefile` saves the cell's content as a file. The file repeats the definitions from Sections 3 to 5 so the folder works on its own.


In [ ]:
%%writefile relapse_agent/agent.py
import re

from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import ToolContext
from google.genai import types

retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

MODEL_NAME = "gemini-3.5-flash-lite"

DEATH_PATTERN = re.compile(
    r"\b(died|dead|death|deceased|passed away|succumbed|fatal(?:ly)?)\b", re.IGNORECASE
)
RELAPSE_PATTERN = re.compile(r"\b(relaps\w*|recurr\w*)\b", re.IGNORECASE)
NEGATION_PATTERN = re.compile(
    r"\b(no|not|without|never|free of|absence of|absent)\b", re.IGNORECASE
)


def find_relapse_death_evidence(tool_context: ToolContext) -> dict:
    """Finds the sentences of the translated clinical text that mention death and relapse.

    Reads the English translation that the TranslatorAgent saved in session state
    under the key 'english_text'. Takes no arguments.

    Returns:
        Dictionary with status and evidence.
        Success: {"status": "success", "death_sentences": [...], "relapse_sentences": [...],
                  "sentences_with_both": [...], "relapse_sentences_possibly_negated": [...]}
        Error: {"status": "error", "error_message": "..."}
    """
    text = tool_context.state.get("english_text")
    if not text:
        return {
            "status": "error",
            "error_message": "No English text found in session state under 'english_text'.",
        }

    sentences = re.split(r"(?<=[.!?;])\s+", text.strip())
    death = [s for s in sentences if DEATH_PATTERN.search(s)]
    relapse = [s for s in sentences if RELAPSE_PATTERN.search(s)]
    both = [s for s in death if RELAPSE_PATTERN.search(s)]
    negated = [s for s in relapse if NEGATION_PATTERN.search(s)]

    return {
        "status": "success",
        "death_sentences": death,
        "relapse_sentences": relapse,
        "sentences_with_both": both,
        "relapse_sentences_possibly_negated": negated,
    }


TRANSLATOR_INSTRUCTION = """You are a professional medical translator.
Translate the Original clinical text written by the user in different languages into English. If the text is already in English, return it as is.

Rules:
- Translate faithfully and completely. Do NOT summarize, interpret, add or omit information.
- Keep negations exactly as written (for example 'pas de rechute' -> 'no relapse').
- Use standard English medical terms (for example 'relapse','recurrence','death', 'died').
- Output ONLY the English translation, with no introduction, notes or quotation marks."""

CLASSIFIER_INSTRUCTION = """You are a careful clinical text screener.

The English clinical text to analyse is:
{english_text}

Question: does this text state that the patient died of a relapse (or recurrence) of their disease?

Steps:
1. Call the `find_relapse_death_evidence` tool (it takes no arguments) to get the sentences that mention death and relapse.
2. Decide using the text above, with the tool output as help.

Answer YES only if the text explicitly says the patient died AND attributes the death to a relapse or recurrence.
Answer NO in every other case, including:
- the patient is alive,
- the patient died of another cause,
- the cause of death is not stated,
- a relapse is mentioned but not stated as the cause of death,
- the relapse is negated (for example 'no sign of relapse', 'did not die of a relapse').

Respond with exactly one line and nothing else:
ANSWER: YES
or
ANSWER: NO"""

translator_agent = LlmAgent(
    name="TranslatorAgent",
    model=Gemini(model=MODEL_NAME, retry_options=retry_config),
    description="Translates Multilingual clinical text into faithful English.",
    instruction=TRANSLATOR_INSTRUCTION,
    output_key="english_text",
)

classifier_agent = LlmAgent(
    name="RelapseDeathClassifierAgent",
    model=Gemini(model=MODEL_NAME, retry_options=retry_config),
    description="Answers whether an English clinical text says the patient died of a relapse.",
    instruction=CLASSIFIER_INSTRUCTION,
    tools=[find_relapse_death_evidence],
    output_key="relapse_verdict",
)

root_agent = SequentialAgent(
    name="RelapseScreeningPipeline",
    description="Translates a Multilingual clinical note and says whether the patient died of a relapse.",
    sub_agents=[translator_agent, classifier_agent],
)

Writing relapse_agent/agent.py


### 10.3: Try it in the ADK web interface (optional, interactive)

The ADK web interface lets you chat with the pipeline and open the **Events** and **Trace** tabs to see each model call and tool call with timings. In a terminal, from the folder that contains `relapse_agent`, run:

```
adk web --log_level DEBUG
```

Then open `http://localhost:8000`, pick `relapse_agent`, paste a clinical note and inspect the trace. Stop the command with Ctrl+C when finished.


---
## Section 11: Evaluation

Observability is **reactive**: you look after something went wrong. **Evaluation is proactive**: you test the agent on known cases, so problems (or a *regression* after a prompt change) are caught before users meet them.

Two things can be measured: the **final response** and the **trajectory** (which tools the agent used). Here the final answer is tested in two ways.

### 11.1: Custom labeled test (the metric that matters most)

Because the correct `YES` or `NO` is known for every sample, the clearest metric is **accuracy on the answer label**. Each note runs in its **own fresh session**, and a minimum accuracy is required to pass.


In [ ]:
def parse_answer(text):
    """Extracts YES/NO from 'ANSWER: YES' (returns None if the format is wrong)."""
    match = re.search(r"ANSWER:\s*(YES|NO)\b", text or "", re.IGNORECASE)
    return match.group(1).upper() if match else None


async def evaluate_pipeline(samples, min_accuracy: float = 0.85):
    svc = InMemorySessionService()
    eval_runner = Runner(agent=build_pipeline(), app_name="relapse_eval_app", session_service=svc)

    correct, failures = 0, []
    for s in samples:
        raw, state = await screen_note(eval_runner, svc, s["Original"], f"eval-{s['id']}")
        predicted = parse_answer(raw)
        ok = predicted == s["expected"]
        correct += ok
        print(f"{'Pass' if ok else 'Did not pass'} {s['id']:<38} expected={s['expected']:<3} got={predicted}")
        if not ok:
            failures.append({"id": s["id"], "english_text": state.get("english_text"), "raw_answer": raw})

    accuracy = correct / len(samples)
    print(f"\nAccuracy: {correct}/{len(samples)} = {accuracy:.0%}   (required: {min_accuracy:.0%})")
    print("RESULT:", "PASS" if accuracy >= min_accuracy else "FAIL")
    return accuracy, failures


accuracy, failures = await evaluate_pipeline(SAMPLES)

Pass relapse_death_explicit                 expected=YES got=YES
Pass death_other_cause_no_relapse           expected=NO  got=NO
Pass relapse_but_alive                      expected=NO  got=NO
Pass death_cause_not_stated                 expected=NO  got=NO
Pass recurrence_synonym                     expected=YES got=YES
Pass negated_relapse                        expected=NO  got=NO
Pass died_after_relapse_no_causal_link      expected=NO  got=NO

Accuracy: 7/7 = 100%   (required: 85%)
RESULT: PASS


### 11.2: Analyse failures

If a case fails, look at the **English translation** first. A wrong answer usually comes from either (a) a translation that lost a negation, or (b) an ambiguous rule in the classifier's instruction. Fix the instruction, rerun, and make sure nothing that used to pass now fails. This is **regression testing**.

In the saved run all seven cases passed, so there is nothing to analyse.


In [ ]:
if not failures:
    print("No failures on this run.")
for f in failures:
    print("Case   :", f["id"])
    print("English:", f["english_text"])
    print("Answer :", f["raw_answer"])
    print()

No failures on this run.


### 11.3: Systematic evaluation with the ADK command line (`adk eval`)

ADK has its own evaluation tool. It works in four steps: **(1)** define the evaluation configuration, **(2)** create test cases, **(3)** run the agent, **(4)** compare the results.

**Step 1: the configuration** (`test_config.json`):

- `tool_trajectory_avg_score: 1.0` means the agent must call `find_relapse_death_evidence` exactly as expected.
- `response_match_score: 0.8` means the final response must be at least 80% similar to the expected one.

Note: because the answer is the short line `ANSWER: YES` or `ANSWER: NO`, text similarity does separate a right answer (score 1.0) from a wrong one (score 0.5). With a long free-text answer, similarity scores could hardly tell YES from NO.


In [ ]:
eval_config = {
    "criteria": {
        "tool_trajectory_avg_score": 1.0,  
        "response_match_score": 0.8,       
    }
}

with open("relapse_agent/test_config.json", "w") as f:
    json.dump(eval_config, f, indent=2)

print("Evaluation configuration created.")

Evaluation configuration created.


**Step 2: the test cases** (`integration.evalset.json`). They are generated from the same `SAMPLES`, so there is one source of truth. For every case the file stores the input, the **expected final response**, and the **expected tool call** (`find_relapse_death_evidence` with no arguments).


In [ ]:
eval_cases = []
for s in SAMPLES:
    eval_cases.append(
        {
            "eval_id": s["id"],
            "conversation": [
                {
                    "user_content": {"parts": [{"text": s["Original"]}]},
                    "final_response": {"parts": [{"text": f"ANSWER: {s['expected']}"}]},
                    "intermediate_data": {
                        "tool_uses": [{"name": "find_relapse_death_evidence", "args": {}}]
                    },
                }
            ],
        }
    )

evalset = {"eval_set_id": "relapse_screening_suite", "eval_cases": eval_cases}

with open("relapse_agent/integration.evalset.json", "w", encoding="utf-8") as f:
    json.dump(evalset, f, indent=2, ensure_ascii=False)

print(f"{len(eval_cases)} evaluation cases written.")

7 evaluation cases written.


**Steps 3 and 4: run and compare.** `adk eval` runs the agent on every case and compares the response and the tool trajectory with the expectations. `--print_detailed_results` shows a per-case breakdown with scores.

If it fails on a case, check which score failed: a low **trajectory** score means the agent did not call the tool as expected, and a low **response match** score means the answer line was different.

The command below is included for completeness and its output is not saved in this notebook. The labeled accuracy test in 11.1 is the main result.


In [ ]:
!adk eval relapse_agent relapse_agent/integration.evalset.json --config_file_path=relapse_agent/test_config.json --print_detailed_results

---
## Section 12: Deployment to Vertex AI Agent Engine

Deployment moves the pipeline from this notebook to a **managed, scalable service** on Google Cloud (see "What is deployment?" at the top). It requires a **Google Cloud project with billing** and several enabled APIs (Vertex AI, Cloud Storage, Logging, Monitoring, Trace and Telemetry).

> **This section is switched off by default.** Set `RUN_DEPLOYMENT = True` only when you have a project. Remember to delete the agent afterwards (section 12.5), because a running agent can cost money.
>
> For anything involving real patient data, the deployment region, data residency and compliance rules (for example GDPR and health-data rules) must be reviewed by your organisation first.

### 12.1: Settings


In [ ]:
RUN_DEPLOYMENT = False  # <-- switch to True to actually deploy

PROJECT_ID = "enter-project-id"  
DEPLOY_REGION = "europe-west1" 

print(f"Deployment enabled: {RUN_DEPLOYMENT}")

### 12.2: Prepare the deployment folder

Agent Engine builds its own environment, so a **separate folder** (`relapse_agent_deploy`) is prepared with:

- `agent.py`: the same pipeline, with a small header that initialises Vertex AI,
- `requirements.txt`: the dependencies,
- `.env`: switches ADK to **Vertex AI** (instead of the AI Studio API key used locally),
- `.agent_engine_config.json`: scaling and resources (`min_instances: 0` scales to zero when idle to save cost).


In [ ]:
deploy_dir = Path("relapse_agent_deploy")
deploy_dir.mkdir(exist_ok=True)

vertex_header = '''import os

import vertexai

vertexai.init(
    project=os.environ["GOOGLE_CLOUD_PROJECT"],
    location=os.environ["GOOGLE_CLOUD_LOCATION"],
)

'''
(deploy_dir / "agent.py").write_text(vertex_header + Path("relapse_agent/agent.py").read_text(encoding="utf-8"), encoding="utf-8")
(deploy_dir / "__init__.py").write_text("from . import agent\n")
(deploy_dir / "requirements.txt").write_text("google-adk\nopentelemetry-instrumentation-google-genai\n")
(deploy_dir / ".env").write_text('GOOGLE_CLOUD_LOCATION="global"\nGOOGLE_GENAI_USE_VERTEXAI=1\n')
(deploy_dir / ".agent_engine_config.json").write_text(
    json.dumps({"min_instances": 0, "max_instances": 1, "resource_limits": {"cpu": "1", "memory": "1Gi"}}, indent=4)
)

print("Deployment folder ready:", sorted(p.name for p in deploy_dir.iterdir()))

Deployment folder ready: ['.agent_engine_config.json', '.env', '__init__.py', 'agent.py', 'requirements.txt']


### 12.3: Deploy

`adk deploy agent_engine` packages the folder, uploads it and creates the deployment. It can take several minutes. Log in to Google Cloud first with `gcloud auth application-default login`.


In [ ]:
if RUN_DEPLOYMENT:
    assert PROJECT_ID != "enter-project-id", "Set PROJECT_ID first."
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

    # Credentials: log in once from a terminal with gcloud auth application-default login
    subprocess.run(
        [
            "adk", "deploy", "agent_engine",
            f"--project={PROJECT_ID}",
            f"--region={DEPLOY_REGION}",
            str(deploy_dir),
            f"--agent_engine_config_file={deploy_dir}/.agent_engine_config.json",
        ],
        check=True,
    )
else:
    print("Skipped (RUN_DEPLOYMENT = False).")


### 12.4: Retrieve and test the deployed agent

The most recently deployed agent is fetched and a note is streamed to it. Several items come back: the model's tool call, the tool response and the final answer.


In [ ]:
if RUN_DEPLOYMENT:
    import vertexai
    from vertexai import agent_engines

    vertexai.init(project=PROJECT_ID, location=DEPLOY_REGION)
    remote_agent = list(agent_engines.list())[0]
    print("Connected to:", remote_agent.resource_name)

    async for item in remote_agent.async_stream_query(message=SAMPLES[0]["Original"], user_id="user_42"):
        print(item)
else:
    print("Skipped (RUN_DEPLOYMENT = False).")

### 12.5: Clean up

Delete the deployed agent when finished, so it does not keep costing money. The cell does nothing unless `RUN_DEPLOYMENT = True`.


In [ ]:
if RUN_DEPLOYMENT:
    remote_agent.delete(force=True)
    print("Deployed agent deleted.")
else:
    print("Skipped (RUN_DEPLOYMENT = False).")


---
## Summary and limitations

**What was built:** a two-agent pipeline. A translator agent converts a clinical note from any language to English, and a classifier agent (with one custom tool) answers whether the text says the patient died of a relapse. Around it: persistent sessions, long-term memory, human approval before an action, logging and metrics, an ADK project folder, an evaluation, and a prepared deployment to Vertex AI Agent Engine.

**Result:** on the seven invented test notes the pipeline answered all seven correctly (7/7, 100%, required minimum 85%).

**Limitations**

- The test set is small and invented, so 100% does not show real-world accuracy. It would need many real, de-identified notes reviewed by clinicians.
- The tool uses simple word patterns (death words, relapse words, negation words). It only supports the model's decision and can miss unusual phrasing.
- A translation error that drops a negation would change the answer, so the translator's output should be checked in important cases.
- The deployment cells were prepared but not run, because they need a billed Google Cloud project.
- This is a screening helper only. A clinician must verify any result that matters.
